# ProjectEcho — Phase 20: PPO Training

**Setup:** In Kaggle, attach the `projectecho-training` dataset (your uploaded `training/` folder) as input data before running.

Trained checkpoints and TensorBoard logs are written to `/kaggle/working/checkpoints/`.
Download `best_model.zip` or `final_model.zip` from the Output tab when done.

In [ ]:
# Install training dependencies from the repo requirements file
import subprocess, sys
TRAINING_DIR = "/kaggle/datasets/maskedkunsiquat/projectecho-training"
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "-r", f"{TRAINING_DIR}/requirements.txt",
], check=True)

In [ ]:
# Make the training/ source files importable
import sys, os
TRAINING_DIR = "/kaggle/datasets/maskedkunsiquat/projectecho-training"
if TRAINING_DIR not in sys.path:
    sys.path.insert(0, TRAINING_DIR)

OUTPUT_DIR = "/kaggle/working/checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("sys.path OK. Output dir:", OUTPUT_DIR)

In [ ]:
# Smoke-test: env imports and resets correctly
from env_wrappers import LeagueEnv
env = LeagueEnv(num_tribes=4, max_ticks=50)
obs, _ = env.reset(seed=42)
mask = env.action_masks()

assert obs.shape == env.observation_space.shape, \
    f"obs shape mismatch: expected {env.observation_space.shape}, got {obs.shape}"
assert mask.shape == (env.action_space.n,), \
    f"mask shape mismatch: expected ({env.action_space.n},), got {mask.shape}"
assert mask[11], "ACTION_REST (index 11) must always be valid after reset"

print(f"Smoke test passed — obs: {obs.shape}, valid actions: {mask.sum()}/{env.action_space.n}")

In [ ]:
# Run training.
#
# Kaggle free tier: 2 CPU cores, 13 GB RAM, 30 GPU hrs/week.
# --envs 4 is safe; drop to 2 if you hit OOM.
# --timesteps 500000 takes ~2-3 hours on P100 GPU.
#
# Resume a previous run by adding:
#   "--resume", "/kaggle/working/checkpoints/best_model",

result = subprocess.run([
    sys.executable,
    os.path.join(TRAINING_DIR, "train.py"),
    "--timesteps", "500000",
    "--envs",      "4",
    "--seed",      "42",
    "--output-dir", OUTPUT_DIR,
    "--win-rate",  "0.60",
], check=False)

if result.returncode != 0:
    raise RuntimeError(f"train.py exited with code {result.returncode} — check output above")

In [ ]:
# List saved checkpoints
for f in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(path) // 1024
    print(f"{f:40s}  {size} KB")